# Sesión 6 · Notebook 4 — Práctica final integradora

Construir en Python un **informe de salud** de OrderFlow que combine, en una sola
vista:

1. **Métricas** de Prometheus: throughput, porcentaje de error y latencia p95.
2. **Logs** de Elasticsearch: errores recientes.
3. Un **veredicto** automático: OK, ATENCIÓN o CRÍTICO.

Es la consolidación del curso entero. Los dos pilares que capturaste en el
Capítulo 1, leídos por código y convertidos en una decisión.

In [ ]:
import requests
from elasticsearch import Elasticsearch

PROM = "http://localhost:9090"
es = Elasticsearch("http://localhost:9200")


def prom_scalar(expr):
    """Valor escalar de una consulta instantánea, o None si no hay datos."""
    r = requests.get(f"{PROM}/api/v1/query", params={"query": expr}, timeout=10)
    res = r.json()["data"]["result"]
    return float(res[0]["value"][1]) if res else None

In [ ]:
# 1) Métricas clave
throughput = prom_scalar("sum(rate(orderflow_orders_processed_total[5m]))")

error_pct = prom_scalar(
    "100 * sum(rate(orderflow_orders_failed_total[5m])) "
    "/ clamp_min(sum(rate(orderflow_orders_processed_total[5m])) "
    "+ sum(rate(orderflow_orders_failed_total[5m])), 0.001)"
)

p95 = prom_scalar(
    "histogram_quantile(0.95, "
    "sum by (le) (rate(orderflow_processing_duration_seconds_bucket[5m])))"
)

# 2) Errores en los logs de la última hora
try:
    errores = es.count(
        index="orderflow-logs-*",
        query={"bool": {"must": [
            {"term": {"level.keyword": "ERROR"}},
            {"range": {"@timestamp": {"gte": "now-1h"}}},
        ]}},
    )["count"]
except Exception as e:
    errores = None
    print("aviso: no se pudo consultar Elasticsearch:", e)

print("métricas y logs recogidos")

In [ ]:
# 3) Veredicto automático
# Los umbrales son los mismos de la alerta de la Sesión 5 y del panel de la
# Sesión 4. Que las tres cosas usen los mismos números no es casualidad: es
# lo que evita que el dashboard, el correo y el informe se contradigan.
def veredicto(error_pct, p95):
    if error_pct is None or p95 is None:
        return "SIN DATOS"
    if error_pct > 10 or p95 > 1.0:
        return "CRÍTICO"
    if error_pct > 5 or p95 > 0.5:
        return "ATENCIÓN"
    return "OK"


estado = veredicto(error_pct, p95)

print("========== INFORME DE SALUD · OrderFlow ==========")
print(f"Throughput (órd/s, 5m) : {throughput:.2f}" if throughput is not None else "Throughput            : s/d")
print(f"% de error (5m)        : {error_pct:.1f}%" if error_pct is not None else "% de error            : s/d")
print(f"Latencia p95 (5m)      : {p95:.3f}s" if p95 is not None else "Latencia p95          : s/d")
print(f"Errores en logs (1h)   : {errores}" if errores is not None else "Errores en logs       : s/d")
print("-" * 50)
print(f"VEREDICTO: {estado}")

## Ejercicio

Amplía este notebook para que:

1. Guarde el informe en `informe_salud.txt`, con fecha y hora.
2. Repita la medición cada minuto durante cinco minutos y muestre la evolución.
3. Envíe una notificación al `webhook-receiver` de la Sesión 5 cuando el
   veredicto sea `CRÍTICO`, cerrando el ciclo: métricas → logs → alerta.

Para el punto 3, el endpoint es `http://localhost:5001/alertas` y acepta un POST
con JSON. Puedes comprobar que llegó con:

```bash
docker compose logs --tail 30 webhook-receiver
```

*Para provocar un veredicto CRÍTICO, sube `ERROR_RATE_PCT` a 30 en tu `.env` y
recrea el processor, igual que en la Sesión 5. Déjalo en 5 al terminar.*